# IFRS S1/S2 Writer Stage

Consumes `generation_blocks_<bank>.json` + `evidence_store_<bank>.json` + `section_plan_<bank>.json`
and produces the aligned report. Per block: build prompt → call Azure → parse the JSON contract →
**deterministic gate** (re-resolve every cited `citation_id` to the store and check the number matches)
→ bounded revision on failure → assemble into section > subsection > block.

Traceability is enforced by code, not trusted: a hallucinated or wrong number cannot pass the gate.

In [ ]:
import os, re, json, time
from pathlib import Path
from collections import defaultdict
try:
    import requests
except ImportError:
    requests = None

OUT = Path("mapping_outputs")          # where the mapper wrote its artifacts
BANK = "BANK01"
MOCK_MODE = False                      # True = offline dry-run (no Azure calls); flip to False for real runs
JSON_MODE = True                       # send response_format={"type":"json_object"} (needs a recent api-version)
TEMPERATURE = 0.2
MAX_TOKENS = 2000
MAX_REVISIONS = 2

# --- load .env (full deployment URL + key) ---
def load_env(path=".env"):
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
load_env()

# accept several common var names; AZURE_OPENAI_URL should be the FULL chat/completions URL incl. ?api-version=
AZURE_URL = next((os.environ[k] for k in
    ("AZURE_OPENAI_URL","AZURE_OPENAI_ENDPOINT","AZURE_OPENAI_CHAT_URL","OPENAI_URL") if k in os.environ), None)
AZURE_KEY = next((os.environ[k] for k in
    ("AZURE_OPENAI_API_KEY","AZURE_OPENAI_KEY","OPENAI_API_KEY","API_KEY") if k in os.environ), None)
if not MOCK_MODE:
    assert AZURE_URL and AZURE_KEY, "Set AZURE_OPENAI_URL (full deployment URL) and AZURE_OPENAI_API_KEY in .env"
print("endpoint:", (AZURE_URL[:60]+"...") if AZURE_URL else "(mock)", "| key:", "set" if AZURE_KEY else "none",
      "| MOCK_MODE:", MOCK_MODE)

In [ ]:
def azure_chat(messages, temperature=TEMPERATURE, max_tokens=MAX_TOKENS, json_mode=JSON_MODE,
               retries=4, timeout=120):
    """POST to the full Azure deployment URL with header api-key. Returns the message content string."""
    if MOCK_MODE:
        return _mock_chat(messages)
    body = {"messages": messages, "temperature": temperature, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    headers = {"api-key": AZURE_KEY, "Content-Type": "application/json"}
    last = None
    for attempt in range(retries):
        try:
            r = requests.post(AZURE_URL, headers=headers, json=body, timeout=timeout)
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"]
            # api-version too old for response_format -> retry once without it
            if r.status_code == 400 and "response_format" in r.text and body.pop("response_format", None):
                continue
            if r.status_code in (429, 500, 502, 503, 504):
                last = RuntimeError(f"{r.status_code}: {r.text[:200]}")
                time.sleep(2 ** attempt); continue
            raise RuntimeError(f"Azure {r.status_code}: {r.text[:300]}")
        except Exception as e:
            last = e; time.sleep(2 ** attempt)
    raise last

def parse_contract(raw):
    s = raw.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\n?", "", s); s = re.sub(r"\n?```$", "", s)
    return json.loads(s)

In [ ]:
# Mock Azure client: cites REAL ids/values from the prompt so the gate exercises the true path.
def _mock_chat(messages):
    user = messages[1]["content"] if len(messages) > 1 else messages[0]["content"]
    is_revision = messages[-1]["content"].startswith(("Fix ALL", "That was not"))
    mode = ("narrative" if "MODE: narrative" in user else "absence" if "MODE: absence" in user else "data_backed")
    reqids = re.findall(r"\[([A-Z0-9_]+)\]", user.split("EVIDENCE")[0])
    ev = re.findall(r"- (E-[A-Z0-9-]+): ([^\n(]+)", user)
    if mode == "narrative" or not ev:
        return json.dumps({"prose": "The entity confirms compliance with the applicable disclosure requirements.",
            "citations_used": [], "numeric_claims": [], "requirements_addressed": reqids[:6],
            "standards_covered": ["IFRS S1"]})
    picks = ev[:2]
    claims, cids, frags = [], [], []
    for cid, val in picks:
        n = re.search(r"-?\d[\d,]*\.?\d*", val)
        cids.append(cid)
        if n:
            claims.append({"text": n.group(0), "citation_id": cid})
            frags.append(f"{n.group(0)} [{cid}]")
        else:
            frags.append(f"the disclosed value [{cid}]")
    return json.dumps({"prose": "For the reporting period, " + "; ".join(frags) + ".",
        "citations_used": cids, "numeric_claims": claims,
        "requirements_addressed": reqids[:8], "standards_covered": ["IFRS S1", "IFRS S2"]})
print("mock client defined (used when MOCK_MODE=True)")

In [ ]:
blocks = json.loads((OUT / f"generation_blocks_{BANK}.json").read_text(encoding="utf-8"))
store  = json.loads((OUT / f"evidence_store_{BANK}.json").read_text(encoding="utf-8"))
plan   = json.loads((OUT / f"section_plan_{BANK}.json").read_text(encoding="utf-8"))
blocks_by_id = {b["block_id"]: b for b in blocks}
SECTION_TITLE = {s["section_key"]: s["section_title"] for s in plan["sections"]}

def deref(path):
    cur = store
    for p in path.strip("/").split("/"):
        cur = cur[p.replace("~1", "/").replace("~0", "~")]
    return cur
print(f"{len(blocks)} blocks | {sum(len(s['subsections']) for s in plan['sections'])} subsections | store {len(store)} collections")

In [ ]:
STYLE_GUIDE = (
 "STYLE: formal regulatory disclosure prose; concise; no marketing language; present figures with their "
 "units and reporting year; integrate IFRS S1 and S2 into one narrative where both apply (do not repeat)."
)  # <- inject your V9.7 style guide text here

MODE_INSTRUCTION = {
 "data_backed": ("Write the disclosure from the evidence. EVERY figure you state MUST be immediately followed "
                 "by its citation in square brackets, e.g. [E-BANK01-0037]. Never state a number that is not "
                 "in the evidence list. Prefer [primary] items for headline figures."),
 "absence":     ("The required data is unavailable. State the absence following the gap instruction verbatim. "
                 "Do NOT report missing values as zero and do NOT invent figures."),
 "narrative":   ("No quantitative data applies. Write a brief qualitative compliance statement addressing the "
                 "requirement(s). Use NO numbers and cite no evidence."),
}

def build_messages(block, section_title):
    reqs = "\n".join(
        f"- [{r['requirement_id']}] ({r['standard']} \u00b6{r['paragraph_id']}{r['clause_path'] or ''}): {r['requirement_text']}"
        for r in block["requirements"])
    def ev_val(e):
        v = e["value"]; return json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v
    evidence = "\n".join(
        f"- {e['citation_id']}: {ev_val(e)} {e['unit'] or ''} ({e['field']}, {e['reporting_year']})"
        f"{' [primary]' if e['is_primary'] else ''}"
        for e in block["evidence"]) or "(no evidence)"
    gaps = "\n".join(f"- {g['field']}: {g.get('instruction','')}" for g in block.get("data_gaps", []))
    system = ("You are an expert sustainability-reporting writer preparing a bank's IFRS S1 and IFRS S2 aligned "
              "climate-related disclosure. You write only what the requirements ask for and only what the evidence "
              "supports. You never invent figures. You cite every figure.")
    user = f"""SECTION: {section_title} > {block['subsection_title']}
CONCEPT: {block['concept']}    STANDARDS: {', '.join(block['standards_present'])}    MODE: {block['generation_mode']}

REQUIREMENTS (produce ONE aligned disclosure satisfying all of them; where IFRS S1 and IFRS S2 both appear, integrate into a single narrative citing both standards):
{reqs}

EVIDENCE (cite ONLY these citation_ids; each maps to a stored, audited value):
{evidence}
{('GAPS:' + chr(10) + gaps) if gaps else ''}
INSTRUCTION: {MODE_INSTRUCTION[block['generation_mode']]}
{STYLE_GUIDE}

Return ONLY a JSON object, no prose outside it:
{{"prose": "disclosure text with inline [E-...] citations after every figure",
  "citations_used": ["E-..."],
  "numeric_claims": [{{"text": "<the number and unit exactly as written in prose>", "citation_id": "E-..."}}],
  "requirements_addressed": ["<requirement_id>", "..."],
  "standards_covered": ["IFRS S1", "IFRS S2"]}}"""
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

In [ ]:
def _to_number(s):
    m = re.search(r"-?\d[\d,]*\.?\d*", str(s))
    return float(m.group(0).replace(",", "")) if m else None

def gate(block, out):
    """Re-resolve every citation to the store; reject invented ids and mismatched numbers."""
    ev = {e["citation_id"]: e for e in block["evidence"]}
    fails = []
    for c in out.get("citations_used", []):
        if c not in ev:
            fails.append(f"cites unknown citation_id {c}")
    for nc in out.get("numeric_claims", []):
        cid = nc.get("citation_id")
        if cid not in ev:
            fails.append(f"numeric claim cites unknown id {cid}"); continue
        claimed, stored = _to_number(nc.get("text")), _to_number(ev[cid]["value"])
        if claimed is None or stored is None:
            continue
        tol = max(abs(stored) * 0.005, 0.01)           # 0.5% tolerance for rounding
        if abs(claimed - stored) > tol:
            fails.append(f"number '{nc.get('text')}' != evidence {ev[cid]['value']} for {cid}")
        # verify the store still backs the citation (pointer integrity at write time)
        if deref(ev[cid]["path"]) != ev[cid]["value"]:
            fails.append(f"evidence pointer drift for {cid}")
    if block["generation_mode"] == "narrative" and out.get("numeric_claims"):
        fails.append("narrative block must contain no numeric claims")
    if block["generation_mode"] == "data_backed" and block["evidence"] and not out.get("citations_used"):
        fails.append("data_backed block produced no citations")
    return (len(fails) == 0, fails)

In [ ]:
def generate_block(block, section_title, max_revisions=MAX_REVISIONS):
    messages = build_messages(block, section_title)
    last_out, last_fails = None, ["no output"]
    for attempt in range(max_revisions + 1):
        raw = azure_chat(messages)
        try:
            out = parse_contract(raw)
        except Exception as e:
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"That was not valid JSON ({e}). Return ONLY the JSON object."}]
            last_fails = [f"invalid JSON: {e}"]; continue
        ok, fails = gate(block, out)
        if ok:
            return {"ok": True, "attempts": attempt, "output": out, "failures": []}
        last_out, last_fails = out, fails
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": "Fix ALL of these and return corrected JSON only:\n- " + "\n- ".join(fails)}]
    return {"ok": False, "attempts": max_revisions, "output": last_out, "failures": last_fails}

In [ ]:
def run_writer(plan, blocks_by_id):
    generated, report_stats = {}, {"ok": 0, "failed": 0, "attempts": 0}
    for s in plan["sections"]:
        for ss in s["subsections"]:
            for bref in ss["blocks"]:
                block = blocks_by_id[bref["block_id"]]
                res = generate_block(block, s["section_title"])
                generated[bref["block_id"]] = res
                report_stats["attempts"] += res["attempts"]
                report_stats["ok" if res["ok"] else "failed"] += 1
                flag = "" if res["ok"] else f"  !! {res['failures']}"
                print(f"  {bref['block_id']:44s} attempts={res['attempts']} ok={res['ok']}{flag}")
    return generated, report_stats

def assemble(plan, generated):
    L = [f"# IFRS S1 & S2 Aligned Climate-Related Disclosure \u2014 {plan['bank_id']}", ""]
    for s in plan["sections"]:
        L += [f"## {s['order']}. {s['section_title']}", ""]
        for ss in s["subsections"]:
            L += [f"### {s['order']}.{ss['subsection_order']} {ss['subsection_title']}", ""]
            for bref in ss["blocks"]:
                res = generated.get(bref["block_id"])
                if res and res["output"]:
                    L += [res["output"]["prose"].strip(), ""]
    return "\n".join(L)

generated, stats = run_writer(plan, blocks_by_id)
print("\nblocks:", stats)
report_md = assemble(plan, generated)
(OUT / f"report_{BANK}.md").write_text(report_md, encoding="utf-8")
# audit sidecar: every citation used across the report, resolvable to a pointer
audit = [{"block_id": bid, "ok": r["ok"], "attempts": r["attempts"],
          "citations": (r["output"] or {}).get("citations_used", []), "failures": r["failures"]}
         for bid, r in generated.items()]
(OUT / f"report_{BANK}_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"wrote {OUT}/report_{BANK}.md  ({len(report_md)} chars) + audit sidecar")